In [ ]:
'''
%pip install pinecone-client pinecone-text
%pip install langchain-pinecone
'''

In [12]:
import time
start = time.time()

from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = CSVLoader('heritage_rag_full.csv', encoding='utf-8')
text_splitter = RecursiveCharacterTextSplitter( 
    chunk_size=1500, 
    chunk_overlap=200
)
document_list = loader.load_and_split(text_splitter=text_splitter)

runtime = time.time() - start

print('문서 쪼개면서 읽는 시간 :', runtime)

문서 쪼개면서 읽는 시간 : 1.8654720783233643


In [3]:
len(document_list)

16280

In [13]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv()

embedding = UpstageEmbeddings(model="solar-embedding-1-large")

# 데이터 처음 업로드할때 사용

In [15]:
%%time
# pinecone vector database
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

pc = Pinecone()
index_name = "upstage-index"

# database = PineconeVectorStore.from_documents(
#     documents=document_list,
#     embedding=embedding,
#     index_name=index_name
# )

CPU times: total: 0 ns
Wall time: 0 ns


# 업로드한 벡터DB 가져올때

In [34]:
database = PineconeVectorStore(
    embedding=embedding, # 질문을 임베딩하여 유사도 검색
    index_name=index_name,
)

# 2. 답변 생성을 위한 Retrieval

In [14]:
query = input("질문 : ")

질문입력ㄴ


In [35]:
# query = "지금 삼성역에 있는데, 근처에 가볼만한 곳이 있어?"
retriever = database.as_retriever(search_kwargs={'k':4})

# 3. 제공되는 prompt를 활용하여 답변 생성

In [36]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain import hub
# prompt = hub.pull("rlm/rag-prompt")

dictionary = ["""
왕실·귀족 계층 : 왕 / 임금, 왕비 / 중전 / 대비, 왕자 / 공주, 대군 / 옹주 / 군 / 군주, 세자 / 세손, 종친 (왕족), 정승 / 영의정 / 좌의정 / 우의정 (삼정승), 판서 / 참판 / 참의 (육조 관리), 대제학 (홍문관 학자)

사대부·문무 관리 : 양반, 문신 (과거 급제 문관), 무신 (무관 / 장군 / 장수), 유생 (서당 / 성균관 유생), 성균관 생원 / 진사, 서리 (하급 문서행정 관직), 향리 (지방행정 실무자), 서얼 (양반과 천민의 혼혈, 중간계층)

직능 계층 (장인, 예술인 등) : 도공 / 도자기 장인, 목수 / 대목장 / 건축 장인, 화원 / 화공 (그림 그리는 사람), 악공 / 악사 (궁중 음악 담당), 백정 (도축과 정육 담당. 낮은 계급), 무당 / 무녀 / 천녀, 광대 (탈춤, 풍물놀이), 필사장 (문헌 필사), 자수장 / 옻칠장 / 소목장 / 전통 기술자, 주역 / 복서 (점치는 사람)

생산·일반 민중 계층 : 농민, 상인 / 보부상, 수공업자 / 염전민 / 어민, 머슴 / 하인, 기생 (풍류와 예술을 담당한 여성 예인층), 여염민 (일반 서민 여성), 화전민 (산간 화전 밭 경작자)

종교 및 학문 계층 : 스님 / 승려, 주지 / 고승 / 대사, 선비, 서당 훈장, 유학자 / 유교 학자, 도사 / 도인 (도교 계열), 천주교 신자 (박해받던 시대 포함), 성직자 (개신교/천주교 포함 근대기 이후)

군사·경비 계층 : 장군 / 무관, 훈련도감 군인, 수군 / 조운수군, 의병 / 민병, 포도청 군사, 금위영 / 어영청 군사

낮은 신분 및 천민 계층 : 노비 / 천민, 공노비 / 사노비, 백정, 광대, 창기 (관기, 관청 소속 기녀), 무당 / 작두무, 거지 / 떠돌이, 사형집행인 (형리)

근대기 포함 특수 계층 : 독립운동가 / 애국지사, 서양인 선교사, 개화파 인사, 통역관 / 역관, 신여성 / 여학교 학생, 항일 의병, 동학 농민군
"""]

prompt = ChatPromptTemplate.from_template(f"""사용자와의 대화 내용을 바탕으로 문화재와 관련된 인물과 직업을 이야기해줘. 
전생 캐릭터를 고를때는 사전을 참고해줘.
사용자의 전생도 재미있게 골라줘. 전생 캐릭터 설명과 짧고 흥미로운 이야기 (300자 이내)를 만들어줘.
그리고 마지막엔 유산 하나 추천해줘. 이유도 간단히 포함해.
사전 : {dictionary}
질문 : {{question}}
[참고 문서]
{{context}}""")

from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4.1-nano")

In [ ]:
# from langchain_core.output_parsers import StrOutputParser
# from langchain_core.prompts import ChatPromptTemplate

# dictionary = ["""
# 왕실·귀족 계층 : 왕 / 임금, 왕비 / 중전 / 대비, 왕자 / 공주, 대군 / 옹주 / 군 / 군주, 세자 / 세손, 종친 (왕족), 정승 / 영의정 / 좌의정 / 우의정 (삼정승), 판서 / 참판 / 참의 (육조 관리), 대제학 (홍문관 학자)

# 사대부·문무 관리 : 양반, 문신 (과거 급제 문관), 무신 (무관 / 장군 / 장수), 유생 (서당 / 성균관 유생), 성균관 생원 / 진사, 서리 (하급 문서행정 관직), 향리 (지방행정 실무자), 서얼 (양반과 천민의 혼혈, 중간계층)

# 직능 계층 (장인, 예술인 등) : 도공 / 도자기 장인, 목수 / 대목장 / 건축 장인, 화원 / 화공 (그림 그리는 사람), 악공 / 악사 (궁중 음악 담당), 백정 (도축과 정육 담당. 낮은 계급), 무당 / 무녀 / 천녀, 광대 (탈춤, 풍물놀이), 필사장 (문헌 필사), 자수장 / 옻칠장 / 소목장 / 전통 기술자, 주역 / 복서 (점치는 사람)

# 생산·일반 민중 계층 : 농민, 상인 / 보부상, 수공업자 / 염전민 / 어민, 머슴 / 하인, 기생 (풍류와 예술을 담당한 여성 예인층), 여염민 (일반 서민 여성), 화전민 (산간 화전 밭 경작자)

# 종교 및 학문 계층 : 스님 / 승려, 주지 / 고승 / 대사, 선비, 서당 훈장, 유학자 / 유교 학자, 도사 / 도인 (도교 계열), 천주교 신자 (박해받던 시대 포함), 성직자 (개신교/천주교 포함 근대기 이후)

# 군사·경비 계층 : 장군 / 무관, 훈련도감 군인, 수군 / 조운수군, 의병 / 민병, 포도청 군사, 금위영 / 어영청 군사

# 낮은 신분 및 천민 계층 : 노비 / 천민, 공노비 / 사노비, 백정, 광대, 창기 (관기, 관청 소속 기녀), 무당 / 작두무, 거지 / 떠돌이, 사형집행인 (형리)

# 근대기 포함 특수 계층 : 독립운동가 / 애국지사, 서양인 선교사, 개화파 인사, 통역관 / 역관, 신여성 / 여학교 학생, 항일 의병, 동학 농민군
# """]

# prompt = ChatPromptTemplate.from_template(f"""사용자와의 대화 내용을 바탕으로 문화재와 관련된 인물과 직업을 이야기해줘. 
# 전생 캐릭터를 고를때는 사전을 참고해줘.
# 사용자의 전생도 재미있게 골라줘. 전생 캐릭터 설명과 짧고 흥미로운 이야기 (300자 이내)를 만들어줘.
# 그리고 마지막엔 유산 하나 추천해줘. 이유도 간단히 포함해.
# 사전 : {dictionary}
# 질문 : {{question}}""")

In [37]:
from langchain.chains import RetrievalQA
qa_chain = RetrievalQA.from_chain_type(
    llm,
    retriever=retriever, # database.as_retriever()
    chain_type_kwargs={"prompt":prompt}
)

In [17]:
# ai_message = qa_chain.invoke({'query':input("질문 : ")})
# ai_message

질문입력광화문 사진 링크좀 줘


{'query': '광화문 사진 링크좀 줘',
 'result': '이곳은 광화문의 사진 링크입니다: [https://www.heritage.go.kr/gung/gogung1/images/ic-c1.jpg](https://www.heritage.go.kr/gung/gogung1/images/ic-c1.jpg)'}

In [38]:
dictionary_chain = prompt | llm | StrOutputParser()
# dictionary_chain.invoke({"queistion":query})

In [39]:
new_chain = {"query":dictionary_chain} | qa_chain

In [41]:
new_chain.invoke({"question":input('질문 : ')})

질문 : 고려청자


KeyError: "Input to ChatPromptTemplate is missing variables {'context'}.  Expected: ['context', 'question'] Received: ['question']\nNote: if you intended {context} to be part of the string and not a variable, please escape it with double curly braces like: '{{context}}'.\nFor troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/INVALID_PROMPT_INPUT "

In [3]:
import time
from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI

# Pinecone 연결 및 벡터 저장소 생성
pc = Pinecone()  
index_name = "upstage-index"

database = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name,
)

retriever = database.as_retriever(search_kwargs={'k': 4})

# Step 3: 전생 신분 사전
dictionary = ["""
왕실·귀족 계층 : 왕 / 임금, 왕비 / 중전 / 대비, 왕자 / 공주, 대군 / 옹주 / 군 / 군주, 세자 / 세손, 종친 (왕족), 정승 / 영의정 / 좌의정 / 우의정 (삼정승), 판서 / 참판 / 참의 (육조 관리), 대제학 (홍문관 학자)
사대부·문무 관리 : 양반, 문신 (과거 급제 문관), 무신 (무관 / 장군 / 장수), 유생 (서당 / 성균관 유생), 성균관 생원 / 진사, 서리 (하급 문서행정 관직), 향리 (지방행정 실무자), 서얼 (양반과 천민의 혼혈, 중간계층)
직능 계층 (장인, 예술인 등) : 도공 / 도자기 장인, 목수 / 대목장 / 건축 장인, 화원 / 화공 (그림 그리는 사람), 악공 / 악사 (궁중 음악 담당), 백정 (도축과 정육 담당. 낮은 계급), 무당 / 무녀 / 천녀, 광대 (탈춤, 풍물놀이), 필사장 (문헌 필사), 자수장 / 옻칠장 / 소목장 / 전통 기술자, 주역 / 복서 (점치는 사람)
생산·일반 민중 계층 : 농민, 상인 / 보부상, 수공업자 / 염전민 / 어민, 머슴 / 하인, 기생 (풍류와 예술을 담당한 여성 예인층), 여염민 (일반 서민 여성), 화전민 (산간 화전 밭 경작자)
종교 및 학문 계층 : 스님 / 승려, 주지 / 고승 / 대사, 선비, 서당 훈장, 유학자 / 유교 학자, 도사 / 도인 (도교 계열), 천주교 신자 (박해받던 시대 포함), 성직자 (개신교/천주교 포함 근대기 이후)
군사·경비 계층 : 장군 / 무관, 훈련도감 군인, 수군 / 조운수군, 의병 / 민병, 포도청 군사, 금위영 / 어영청 군사
낮은 신분 및 천민 계층 : 노비 / 천민, 공노비 / 사노비, 백정, 광대, 창기 (관기, 관청 소속 기녀), 무당 / 작두무, 거지 / 떠돌이, 사형집행인 (형리)
근대기 포함 특수 계층 : 독립운동가 / 애국지사, 서양인 선교사, 개화파 인사, 통역관 / 역관, 신여성 / 여학교 학생, 항일 의병, 동학 농민군
"""]

# LLM 프롬프트 설정 (전생 캐릭터 + 유산 추천)
prompt = ChatPromptTemplate.from_template(f"""
사용자와의 대화 내용을 바탕으로 문화재와 관련된 인물과 직업을 이야기해줘.
전생 캐릭터를 고를 때는 아래 사전을 참고해줘.
전생 캐릭터 설명과 짧고 흥미로운 이야기 (500자 이내)를 만들어줘.
그리고 마지막엔 문화유산 하나 추천해줘. 추천 이유도 간단히 알려줘.

[사전 정보]
{dictionary}

[참고 문서]
{{context}}

[사용자 질문]
{{question}}

[답변]
""")

# LLM 모델 (GPT-4-nano 사용)
llm = ChatOpenAI(model="gpt-4.1-nano")

# 전생 설명 + 유산 추천 chain
dictionary_chain = prompt | llm | StrOutputParser()

# RAG 기반 문서 검색 chain
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type_kwargs={"prompt": prompt}
)

# 실행
query = "나는 전생에 어떤 사람이었을까?"

print("전생 캐릭터 생성 중...")
result = dictionary_chain.invoke({"question": query})
print("결과:\n", result)

rag_result = qa_chain.run(query)
print(rag_result)


PineconeConfigurationError: You haven't specified an API key. Please either set the PINECONE_API_KEY environment variable or pass the 'api_key' keyword argument to the Pinecone client constructor.

In [4]:
import time
from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda, RunnableMap
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings

load_dotenv()
embedding = UpstageEmbeddings(model="solar-embedding-1-large")

# Pinecone 설정 + Vector 저장소
pc = Pinecone()
index_name = "upstage-index"

database = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name,
)
retriever = database.as_retriever(search_kwargs={'k': 4})

# RAG용 프롬프트 (문서 기반 질문 응답)
prompt = ChatPromptTemplate.from_template("""
당신은 한국 문화유산에 대한 전문 해설사입니다.
다음 문서를 참고하여 사용자 질문에 정성스럽고 흥미롭게 답변해주세요.
답변은 300자 이내로 요약하며, 관련 문화유산 이름과 추천 이유도 간단히 포함해주세요.
대화 내용을 참고해서 사용자의 전생을 추측하고, 전생 캐릭터 설명과 재밌는 이야기도 해줘.


[참고 문서]
{context}

[질문]
{question}

[답변]
""")

llm = ChatOpenAI(model="gpt-4.1-nano")

# RetrievalQA 체인 구성 (문서 기반)
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": prompt}
)

# 후처리 랭체인 (응답 포맷 처리 등 추가 가능)
def format_response(rag_result: str) -> str:
    return f"""문화유산 기반 응답\n\n{rag_result}"""

# 전체 체인 연결 (RAG → 후처리)
full_chain = (
    RunnableLambda(lambda x: qa_chain.run(x["question"])) |
    RunnableLambda(lambda r: format_response(r))
)

# 실행
query = "광화문"
result = full_chain.invoke({"question": query})
print(result)


C:\Users\Admin\AppData\Local\Temp\ipykernel_21928\2742914362.py:60: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  RunnableLambda(lambda x: qa_chain.run(x["question"])) |


문화유산 기반 응답

광화문은 조선시대 경복궁 남문으로, 한국의 역사와 문화의 상징입니다. 임진왜란, 일제 강점기 등 많은 역경을 견뎌냈으며, 현재는 1968년 재건 후 원형을 복원하여 우리 민족의 자긍심을 보여줍니다. 추천 문화유산은 '경복궁 근정문 및 행각'과 '경복궁 근정문'입니다. 광화문을 통해 조선의 위엄과 역사적 힘을 느껴보세요. 전생에는 왕실 관계자였거나 문화유산 수호자였을 캐릭터일지도 몰라요!


In [ ]:
# CUDA 11.8
conda install pytorch torchvision torchaudio pytorch-cuda=11.8 -c pytorch -c nvidia

In [ ]:
# gpu 설정 쿠다버전
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [2]:
#!pip install torch
#conda install pytorch torchvision torchaudio cpuonly -c pytorch

In [44]:
# !pip install transformers torch torchvision pillow
# pip install sentence-transformers
# pip install transformers

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 1.7/1.7 MB 10.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/216.1 MB ? eta -:--:--
   ---------------------------------------- 2.4/216.1 MB 12.2 MB/s eta 0:00:18
    --------------------------------------- 5.0/216.1 MB 12.1 MB/s eta 0:00:18
   - -------------------------------------- 7.6/216.1 MB 12.1 MB/s eta 0:00:18
   - -------------------------------------- 10.2/216.1 MB 11.8 MB/s eta 0:00:18
   -- ------------------------------------- 12.6/216.1 MB 12.0 MB/s eta 0:00:18
   -- ------------------------------------- 15.2/216.1 MB 11.8 MB/s eta 0:00:18
   --- ------------------------------------ 17.8/216.1 MB 11.8 MB/s eta 0:00:17
   --- ------------------------------------ 20.4/216.1 MB 11.9 MB/s eta 0:00:17
   ---- ----------------------------------- 22.8/216.1 MB 11.8 MB/s eta 0:00:17
   -

In [ ]:
pip uninstall numpy -y
pip install numpy --upgrade

In [5]:
# !pip install transformers torch torchvision pillow

import time
import os
import torch
from PIL import Image
from typing import Dict, Optional

# 기존 imports
from langchain_community.document_loaders import CSVLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain.chains import RetrievalQA
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnableLambda, RunnableMap
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings

# CLIP imports 추가
from transformers import CLIPProcessor, CLIPModel

print("라이브러리 로딩 완료!")

# 2. 환경 설정 (기존과 동일)
load_dotenv()
embedding = UpstageEmbeddings(model="solar-embedding-1-large")

# 3. CLIP 모델 초기화
print("CLIP 모델 로딩 중... (처음에는 시간이 걸릴 수 있습니다)")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
print("CLIP 모델 로딩 완료!")

# 4. Pinecone 설정
pc = Pinecone()
index_name = "upstage-index"
database = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name,
)
retriever = database.as_retriever(search_kwargs={'k': 4})

# 5. 문화재 카테고리 매핑 (CLIP 분석용)
HERITAGE_CATEGORIES = {
    "Korean traditional palace building": {
        "korean": "궁궐",
        "keywords": ["궁궐", "궁전", "조선왕조", "경복궁", "창덕궁", "광화문", "대한문"],
        "description": "조선시대 왕궁 건축물"
    },
    "Korean Buddhist temple architecture": {
        "korean": "사찰",
        "keywords": ["사찰", "절", "불교", "대웅전", "법당", "산사", "템플"],
        "description": "불교 사원 건축물"
    },
    "Korean traditional house hanok": {
        "korean": "한옥",
        "keywords": ["한옥", "전통가옥", "기와집", "초가집", "민가", "전통건축"],
        "description": "한국 전통 주거 건축"
    },
    "Korean stone pagoda tower": {
        "korean": "석탑",
        "keywords": ["석탑", "탑", "다층탑", "불탑", "3층석탑", "5층석탑", "석조탑"],
        "description": "석조 불교 탑"
    },
    "Korean Buddha statue sculpture": {
        "korean": "불상",
        "keywords": ["불상", "부처님", "불교조각", "석불", "마애불", "금동불"],
        "description": "불교 조각상"
    },
    "Korean traditional pottery ceramics": {
        "korean": "도자기",
        "keywords": ["도자기", "청자", "백자", "분청사기", "고려청자", "조선백자"],
        "description": "전통 도자기"
    },
    "Korean traditional painting artwork": {
        "korean": "전통회화",
        "keywords": ["전통회화", "민화", "문인화", "산수화", "초상화", "불화"],
        "description": "한국 전통 그림"
    },
    "Korean fortress wall castle": {
        "korean": "성곽",
        "keywords": ["성곽", "성벽", "산성", "읍성", "행궁", "성문"],
        "description": "방어용 성곽 건축"
    }
}

# 6. CLIP 이미지 분석 함수
def analyze_image_with_clip(image_path: str) -> Dict:
    """
    CLIP을 사용하여 이미지에서 문화재 유형을 분석
    
    Args:
        image_path: 분석할 이미지 파일 경로
    
    Returns:
        분석 결과 딕셔너리
    """
    try:
        print(f"이미지 분석 중: {image_path}")
        
        # 이미지 로드 및 검증
        if not os.path.exists(image_path):
            raise FileNotFoundError(f"이미지 파일을 찾을 수 없습니다: {image_path}")
        
        image = Image.open(image_path)
        if image.mode != 'RGB':
            image = image.convert('RGB')
        
        print(f"이미지 크기: {image.size}")
        
        # CLIP 분석용 카테고리 리스트
        categories = list(HERITAGE_CATEGORIES.keys())
        
        # CLIP 처리
        inputs = clip_processor(
            text=categories, 
            images=image, 
            return_tensors="pt", 
            padding=True
        )
        
        with torch.no_grad():
            outputs = clip_model(**inputs)
        
        # 유사도 계산
        logits_per_image = outputs.logits_per_image
        probs = logits_per_image.softmax(dim=1)
        
        # 상위 3개 결과 추출
        top3_indices = probs[0].topk(3).indices
        top3_probs = probs[0].topk(3).values
        
        results = []
        for i, (idx, prob) in enumerate(zip(top3_indices, top3_probs)):
            category_key = categories[idx.item()]
            category_info = HERITAGE_CATEGORIES[category_key]
            confidence = prob.item() * 100
            
            results.append({
                "rank": i + 1,
                "english_category": category_key,
                "korean_category": category_info["korean"],
                "description": category_info["description"],
                "confidence": confidence,
                "keywords": category_info["keywords"]
            })
        
        # 최고 예측 결과
        best_result = results[0]
        
        print(f"분석 결과: {best_result['korean_category']} ({best_result['confidence']:.1f}%)")
        print(f"상위 3개: {[r['korean_category'] for r in results]}")
        
        return {
            "success": True,
            "best_prediction": best_result,
            "top3_predictions": results,
            "search_keywords": best_result["keywords"][:4],  # 상위 4개 키워드
            "search_query": " ".join(best_result["keywords"][:3])  # 검색용 쿼리
        }
        
    except Exception as e:
        print(f"이미지 분석 오류: {e}")
        return {
            "success": False,
            "error": str(e),
            "search_keywords": ["한국", "문화재", "유산"],
            "search_query": "한국 문화재"
        }

# 7. 기존 텍스트용 프롬프트 (약간 수정)
text_prompt = ChatPromptTemplate.from_template("""
당신은 한국 문화유산에 대한 전문 해설사입니다.
다음 문서를 참고하여 사용자 질문에 정성스럽고 흥미롭게 답변해주세요.
답변은 300자 이내로 요약하며, 관련 문화유산 이름과 추천 이유도 간단히 포함해주세요.

[참고 문서]
{context}

[질문]
{question}

[답변]
""")

# 8. 이미지 분석용 특별 프롬프트
image_prompt = ChatPromptTemplate.from_template("""
당신은 한국 문화유산에 대한 전문 해설사입니다.
업로드된 이미지를 AI가 분석한 결과와 관련 문서를 바탕으로 답변해주세요.

[이미지 AI 분석 결과]
- 문화재 유형: {heritage_type}
- 상세 설명: {heritage_description}
- 분석 신뢰도: {confidence:.1f}%
- 관련 키워드: {keywords}

[관련 문서]
{context}

[사용자 질문]
{question}

위 정보를 바탕으로 다음을 포함하여 흥미롭게 설명해주세요:
1. 이 문화재의 역사적 배경과 의미
2. 주요 특징과 구조적 특성  
3. 문화적 가치와 현재 상황
4. 관련된 재미있는 이야기나 전설

답변은 400자 내외로 작성해주세요.

[답변]
""")

# 9. LLM 설정 (기존과 동일)
llm = ChatOpenAI(model="gpt-4o-mini")  # gpt-4.1-nano 대신 안정적인 모델 사용

# 10. 텍스트 기반 RAG 함수 (기존 방식 유지)
def query_with_text(question: str) -> str:
    """기존 텍스트 기반 RAG 쿼리"""
    print(f"텍스트 쿼리: {question}")
    
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        retriever=retriever,
        chain_type="stuff",
        chain_type_kwargs={"prompt": text_prompt}
    )
    
    result = qa_chain.run(question)
    return f"문화유산 기반 응답\n\n{result}"

# 11. 이미지 기반 RAG 함수 (새로 추가)
def query_with_image(image_path: str, question: str = None) -> Dict:
    """
    이미지와 텍스트를 함께 사용한 RAG 쿼리
    
    Args:
        image_path: 분석할 이미지 경로 
        question: 사용자 질문 (선택사항)
    
    Returns:
        분석 및 응답 결과
    """
    
    print(f"\n이미지 기반 문화재 분석 시작!")
    print("=" * 50)
    
    # 1. CLIP으로 이미지 분석
    clip_result = analyze_image_with_clip(image_path)
    
    if not clip_result["success"]:
        return {
            "success": False,
            "error": clip_result["error"]
        }
    
    # 2. CLIP 결과를 바탕으로 벡터 검색
    search_query = clip_result["search_query"]
    print(f"벡터 검색 쿼리: '{search_query}'")
    
    try:
        # Pinecone에서 관련 문서 검색
        docs = database.similarity_search(search_query, k=4)
        context = "\n\n".join([doc.page_content for doc in docs])
        print(f"검색된 관련 문서: {len(docs)}개")
        
        # 3. 사용자 질문 처리
        if not question:
            question = f"이 {clip_result['best_prediction']['korean_category']}에 대해 자세히 설명해주세요."
        
        print(f"최종 질문: {question}")
        
        # 4. 이미지 특화 프롬프트로 LLM 실행
        best_pred = clip_result['best_prediction']
        formatted_prompt = image_prompt.format(
            heritage_type=best_pred['korean_category'],
            heritage_description=best_pred['description'],
            confidence=best_pred['confidence'],
            keywords=", ".join(best_pred['keywords'][:5]),
            context=context,
            question=question
        )
        
        print("AI 응답 생성 중...")
        response = llm.invoke(formatted_prompt)
        
        return {
            "success": True,
            "clip_analysis": clip_result,
            "llm_response": response.content,
            "retrieved_docs": len(docs),
            "final_question": question
        }
        
    except Exception as e:
        print(f"RAG 처리 오류: {e}")
        return {
            "success": False,
            "error": f"RAG 처리 중 오류: {e}",
            "clip_analysis": clip_result
        }

# 12. 기존 후처리 함수 (유지)
def format_response(rag_result: str) -> str:
    return f"""문화유산 기반 응답\n\n{rag_result}"""

# 13. 기존 전체 체인 (텍스트용, 유지)
full_text_chain = (
    RunnableLambda(lambda x: query_with_text(x["question"])) |
    RunnableLambda(lambda r: format_response(r))
)

print("\nRAG + CLIP 시스템 초기화 완료!")
print("=" * 50)

# 14. 테스트 실행 부분
if __name__ == "__main__":
    print("\n시스템 테스트 시작")
    
    # 테스트 1: 기존 텍스트 쿼리
    print("\n기존 텍스트 RAG 테스트:")
    query = "광화문"
    try:
        result = full_text_chain.invoke({"question": query})
        print(result)
    except Exception as e:
        print(f"텍스트 쿼리 오류: {e}")
    
    print("\n새로운 이미지 RAG 테스트:")
    

    IMAGE_PATH = "gbk.png"  
    USER_QUESTION = "이 문화재는 언제 만들어졌나요?" 
    
    if IMAGE_PATH != "gbk.png":
        try:
            image_result = query_with_image(IMAGE_PATH, USER_QUESTION)
            
            if image_result["success"]:
                print("\nCLIP 분석 결과:")
                best_pred = image_result["clip_analysis"]["best_prediction"]
                print(f"   - 문화재 유형: {best_pred['korean_category']}")
                print(f"   - 신뢰도: {best_pred['confidence']:.1f}%")
                print(f"   - 설명: {best_pred['description']}")
                
                print(f"\nAI 최종 응답:")
                print(f"   {image_result['llm_response']}")
                
                print(f"\n추가 정보:")
                print(f"   - 검색된 문서: {image_result['retrieved_docs']}개")
                print(f"   - 사용된 키워드: {image_result['clip_analysis']['search_keywords']}")
            else:
                print(f"이미지 분석 실패: {image_result['error']}")
                
        except Exception as e:
            print(f"이미지 쿼리 오류: {e}")
    else:
        print("IMAGE_PATH를 실제 이미지 경로로 바꿔주세요!")
    
    print("\n테스트 완료!")
    
    # 사용법 안내
    print("\n" + "="*60)
    print("사용법 가이드:")
    print("="*60)
    print("1. 텍스트 질문:")
    print("   result = query_with_text('불국사에 대해 알려주세요')")
    print()
    print("2. 이미지 분석:")
    print("   result = query_with_image('이미지경로.jpg', '이 건물은 언제 지어졌나요?')")
    print()
    print("3. 이미지만으로 분석 (질문 없이):")
    print("   result = query_with_image('이미지경로.jpg')")

C:\Users\Admin\anaconda3\envs\llm\lib\site-packages\torch\_subclasses\functional_tensor.py:295: UserWarning: Failed to initialize NumPy: DLL load failed while importing _multiarray_umath: 지정된 모듈을 찾을 수 없습니다. (Triggered internally at C:\cb\pytorch_1000000000000\work\torch\csrc\utils\tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))
C:\Users\Admin\AppData\Roaming\Python\Python310\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: DLL load failed while importing _multiarray_umath: 지정된 모듈을 찾을 수 없습니다.

ImportError: DLL load failed while importing _multiarray_umath: 지정된 모듈을 찾을 수 없습니다.

ImportError: DLL load failed while importing _multiarray_umath: 지정된 모듈을 찾을 수 없습니다.

ImportError: _multiarray_umath failed to import

ImportError: numpy._core.umath failed to import

# 로드 테스트

In [6]:
from dotenv import load_dotenv
load_dotenv()

FILE_PATH = 'test.pdf'

In [7]:
# 메타데이터 확인용
def show_metadata(docs):
    if docs:
        print("[metadata]")
        print(list(docs[0].metadata.keys()))
        print('\n[examples]')
        max_key_lenth = max(len(k) for k in docs[0].metadata.keys())
        for k, v in docs[0].metadata.items():
            print(f'{k:{max_key_lenth}} : {v}')

In [1]:
!pip install -qU pypdf

In [7]:
# PDF 문서 배열 로드, 문서마다 page 번호 + page 내용
# pip install -qU pypdf
from langchain.document_loaders import PyPDFLoader

loader = PyPDFLoader(FILE_PATH)
docs = loader.load()
print(docs[0].page_content[:300])

# 메타데이터
show_metadata(docs)

국가자격 부정행위 예방 캠페인 : ‘부정행위,  묵인하면 계속됩니다’ .
시험명 2025년 정기 기능사 3회
수험번호 3000567 9 시험구분 실기
종목명 정보처리기능사 선택분야 선택분야없음
성명 고정현 생년월일 1988년 04월 01일
시험일시 및
장소
※ 시작시간 이후 입실 및 응시가 불가하며,  시험장 위치 날짜 입실가능 시간을 확인하시기 바랍니다.
합격(예정)자
발표일자2025년 09월 30일 (화) 09:00
자격증
신청 방법큐넷(www .Q-Net.or .kr) -
자격증/확인서발급에서 신청
검정수수료
환불안내2025년
[metadata]
['producer', 'creator', 'creationdate', 'title', 'moddate', 'source', 'total_pages', 'page', 'page_label']

[examples]
producer     : Skia/PDF m138
creator      : Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36
creationdate : 2025-07-28T02:35:52+00:00
title        : 기능사 수험표 유의사항_250527 작업형 계산기 지참 및 필답형 화장실
moddate      : 2025-07-28T02:35:52+00:00
source       : test.pdf
total_pages  : 6
page         : 0
page_label   : 1


In [5]:
%pip install -qU pymupdf

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [4]:
# OCR
# pip install -qU rapidocr-onnxruntime
# PyMuPDF : 속도 최적화, 페이지에 대한 자세한 메타데이터 포함, 페이지 -> 하나의 문서 반환
# pip install -qU pymupdf
from langchain_community.document_loaders import PyMuPDFLoader
loader = PyMuPDFLoader(FILE_PATH)
docs = loader.load()
print(docs[0].page_content[:300])
show_metadata(docs)

국가자격 부정행위 예방 캠페인 : ‘부정행위, 묵인하면 계속됩니다’.
시험명
2025년 정기 기능사 3회
수험번호
30005679
시험구분
실기
종목명
정보처리기능사
선택분야
선택분야없음
성명
고정현
생년월일
1988년 04월 01일
시험일시 및
장소
※ 시작시간 이후 입실 및 응시가 불가하며, 시험장 위치 날짜 입실가능 시간을 확인하시기 바랍니다.
합격(예정)자
발표일자
2025년 09월 30일 (화) 09:00
자격증
신청 방법
큐넷(www.Q-Net.or.kr) -
자격증/확인서발급에서 신청
검정수수료
환불안내
2025년 07
[metadata]
['producer', 'creator', 'creationdate', 'source', 'file_path', 'total_pages', 'format', 'title', 'author', 'subject', 'keywords', 'moddate', 'trapped', 'modDate', 'creationDate', 'page']

[examples]
producer     : Skia/PDF m138
creator      : Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/138.0.0.0 Safari/537.36
creationdate : 2025-07-28T02:35:52+00:00
source       : test.pdf
file_path    : test.pdf
total_pages  : 6
format       : PDF 1.4
title        : 기능사 수험표 유의사항_250527 작업형 계산기 지참 및 필답형 화장실
author       : 
subject      : 
keywords     : 
moddate      : 2025-07-28T02:35:52+00:00
trapped      : 
modDate      : D:20250728023552+00'00'
crea

In [4]:
%pip install -qU unstructured

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [4]:
%pip install -qU pdfminer

Note: you may need to restart the kernel to use updated packages.


  DEPRECATION: Building 'pdfminer' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'pdfminer'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [4]:
%pip install pdfminer.six==20221105

   ---------------------------------------- 0.0/5.6 MB ? eta -:--:--
   ---------------- ----------------------- 2.4/5.6 MB 12.2 MB/s eta 0:00:01
   ----------------------------------- ---- 5.0/5.6 MB 12.1 MB/s eta 0:00:01
   ---------------------------------------- 5.6/5.6 MB 11.4 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install pi_heif

   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------------- ---------------------- 0.8/1.8 MB 5.6 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 6.8 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [4]:
%pip install "unstructured[inference]"

Note: you may need to restart the kernel to use updated packages.


In [5]:
%pip install unstructured-inference


  Using cached opencv_python-4.12.0.88-cp37-abi3-win_amd64.whl.metadata (19 kB)
  Using cached accelerate-1.9.0-py3-none-any.whl.metadata (19 kB)
  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl.metadata (60 kB)
Using cached opencv_python-4.12.0.88-cp37-abi3-win_amd64.whl (39.0 MB)
Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl (12.9 MB)
   ---------------------------------------- 0.0/15.8 MB ? eta -:--:--
   ----- ---------------------------------- 2.4/15.8 MB 12.2 MB/s eta 0:00:02
   ------------ --------------------------- 5.0/15.8 MB 12.1 MB/s eta 0:00:01
   ------------------- -------------------- 7.6/15.8 MB 12.1 MB/s eta 0:00:01
   ----------------------- ---------------- 9.4/15.8 MB 12.0 MB/s eta 0:00:01
   ---------------------------- ----------- 11.3/15.8 MB 10.5 MB/s eta 0:00:01
   ----------------------------------- ---- 13.9/15.8 MB 10.8 MB/s eta 0:00:01
   ---------------------------------------  15.7/15.8 MB 10.9 MB/s eta 0:00:01
   -----------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.6 which is incompatible.
pinecone-text 0.10.0 requires numpy<2.0,>=1.21.5; python_version < "3.12", but you have numpy 2.2.6 which is incompatible.


In [9]:
%pip install numpy==1.26.4

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'c:\\users\\admin\\anaconda3\\envs\\llm\\lib\\site-packages\\numpy-1.26.4.dist-info\\METADATA'



In [8]:
%pip uninstall numpy

Found existing installation: numpy 1.26.4
Note: you may need to restart the kernel to use updated packages.


error: uninstall-no-record-file

Cannot uninstall numpy 1.26.4

The package's contents are unknown: no RECORD file was found for numpy.

hint: You might be able to recover from this via: pip install --force-reinstall --no-deps numpy==1.26.4


In [4]:
%pip install pdf2image

Note: you may need to restart the kernel to use updated packages.


In [8]:
from langchain_community.document_loaders import UnstructuredPDFLoader

# 1. PDF 불러오기
loader = UnstructuredPDFLoader(FILE_PATH)
docs = loader.load()

# 2. 문서 내용 일부 출력 (앞부분 3개)
for i, doc in enumerate(docs[:3]):
    print(f"\n문서 {i + 1}")
    print(doc.page_content)

# 3. 메타데이터 확인
for i, doc in enumerate(docs[:3]):
    print(f"\n메타데이터 {i + 1}")
    print(doc.metadata)


Error importing huggingface_hub.file_download: 'NoneType' object is not subscriptable


TypeError: 'NoneType' object is not subscriptable

In [4]:
# Unstructured : MD, PDF 비구조화 혹은 반구조화 파일 다루는데에 특화된 인터페이스
# !pip install -qU unstructured
# Unstructured mode="elements" 지정하면 청크 단위로 반환
from langchain_community.document_loaders import UnstructuredPDFLoader
loader = UnstructuredPDFLoader(FILE_PATH, mode="elements")
docs = loader.load()
print(docs.page_content)

# 데이터 카테고리 추출
set(doc.metadata["category"] for doc in docs)
show_metadata(docs)

Error importing huggingface_hub.file_download: 'NoneType' object is not subscriptable


TypeError: 'NoneType' object is not subscriptable

# 전체 테스트

In [ ]:
from langchain_community.document_loaders import PyMuPDFLoader
from docx import Document
from PIL import Image
import os
import base64
import fitz  # PyMuPDF

def extract_pdf_to_markdown_and_docx(pdf_path: str, docx_output_path: str):
    # 1. 텍스트 & 이미지 추출
    loader = PyMuPDFLoader(pdf_path)
    docs = loader.load()

    # 2. Word 문서 객체 생성
    doc = Document()
    doc.add_heading("자동 생성 보고서", 0)

    # 3. 페이지 단위로 반복
    for i, page in enumerate(docs):
        doc.add_heading(f"페이지 {i+1}", level=1)
        doc.add_paragraph(page.page_content.strip())  # 텍스트

        # 4. 이미지 추출
        page_images = extract_images_from_pdf_page(pdf_path, i)
        for img_path in page_images:
            doc.add_picture(img_path, width=None)
            os.remove(img_path)  # 임시 이미지 삭제

    # 5. 저장
    doc.save(docx_output_path)
    print(f"[완료] {docx_output_path} 저장됨")


def extract_images_from_pdf_page(pdf_path, page_num):
    # PyMuPDF를 사용하여 이미지 추출
    doc = fitz.open(pdf_path)
    page = doc[page_num]
    image_list = page.get_images(full=True)
    saved_images = []
    for idx, img in enumerate(image_list):
        xref = img[0]
        base_image = doc.extract_image(xref)
        image_bytes = base_image["image"]
        img_ext = base_image["ext"]
        img_path = f"page_{page_num+1}_img_{idx+1}.{img_ext}"
        with open(img_path, "wb") as f:
            f.write(image_bytes)
        saved_images.append(img_path)
    return saved_images


# pdf

In [2]:
%pip install pdfplumber

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unstructured-inference 1.0.5 requires numpy, which is not installed.



  Using cached pdfplumber-0.11.7-py3-none-any.whl.metadata (42 kB)
  Using cached pdfminer_six-20250506-py3-none-any.whl.metadata (4.2 kB)
   ---------------------------------------- 0.0/5.6 MB ? eta -:--:--
   ---------------- ----------------------- 2.4/5.6 MB 12.2 MB/s eta 0:00:01
   ----------------------------------- ---- 5.0/5.6 MB 11.6 MB/s eta 0:00:01
   ---------------------------------------- 5.6/5.6 MB 11.1 MB/s  0:00:00

  Attempting uninstall: pdfminer.six

    Found existing installation: pdfminer.six 20221105

    Uninstalling pdfminer.six-20221105:

   ---------------------------------------- 0/2 [pdfminer.six]
      Successfully uninstalled pdfminer.six-20221105
   ---------------------------------------- 0/2 [pdfminer.six]
   ---------------------------------------- 0/2 [pdfminer.six]
   ---------------------------------------- 0/2 [pdfminer.six]
   ---------------------------------------- 0/2 [pdfminer.six]
   ---------------------------------------- 0/2 [pdfminer.s

In [9]:
%pip install langchain-community pdfplumber pymupdf python-docx

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: 'c:\\users\\admin\\anaconda3\\envs\\llm\\lib\\site-packages\\numpy-1.26.4.dist-info\\METADATA'



In [1]:
import os
import fitz  # PyMuPDF
import pdfplumber
from langchain_community.document_loaders import PDFPlumberLoader
from docx import Document
from docx.shared import Inches
from tempfile import NamedTemporaryFile

def extract_pdf_with_plumber_and_images(pdf_path: str, output_docx_path: str):
    # Word 문서 시작
    doc = Document()
    doc.add_heading("자동 생성 보고서", level=0)

    # PDFPlumber로 텍스트 + 테이블 추출
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            doc.add_heading(f'페이지 {i + 1}', level=1)

            # 1. 텍스트
            text = page.extract_text()
            if text:
                doc.add_paragraph(text.strip())

            # 2. 테이블
            tables = page.extract_tables()
            for t_idx, table in enumerate(tables):
                doc.add_paragraph(f"[표 {t_idx + 1}]", style='Intense Quote')
                if table:
                    table_doc = doc.add_table(rows=1, cols=len(table[0]))
                    table_doc.style = 'Table Grid'
                    hdr_cells = table_doc.rows[0].cells
                    for j, heading in enumerate(table[0]):
                        hdr_cells[j].text = heading if heading else ""
                    for row_data in table[1:]:
                        row_cells = table_doc.add_row().cells
                        for j, cell in enumerate(row_data):
                            row_cells[j].text = cell if cell else ""

    # PyMuPDF로 이미지 추출
    image_paths = extract_images_from_pdf(pdf_path)

    if image_paths:
        doc.add_page_break()
        doc.add_heading("이미지 목록", level=1)
        for i, img_path in enumerate(image_paths):
            doc.add_paragraph(f"이미지 {i + 1}")
            doc.add_picture(img_path, width=Inches(4.5))
            os.remove(img_path)

    # 저장
    doc.save(output_docx_path)
    print(f"[보고서 생성 완료] {output_docx_path}")



def extract_images_from_pdf(pdf_path: str):
    doc = fitz.open(pdf_path)
    saved_images = []
    for page_index in range(len(doc)):
        page = doc[page_index]
        images = page.get_images(full=True)
        for img_index, img in enumerate(images):
            xref = img[0]
            base_image = doc.extract_image(xref)
            image_bytes = base_image["image"]
            ext = base_image["ext"]
            image_filename = f"page_{page_index+1}_img_{img_index+1}.{ext}"
            with open(image_filename, "wb") as f:
                f.write(image_bytes)
            saved_images.append(image_filename)
    return saved_images


In [2]:
extract_pdf_with_plumber_and_images("test.pdf", "output_report.docx")

Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats


[보고서 생성 완료] output_report.docx


# pdf 찐

In [13]:
import os
import fitz  # PyMuPDF
import pdfplumber
from docx import Document
from docx.shared import Inches

def extract_pdf_all_in_order(pdf_path: str, output_docx_path: str):
    # Word 문서 시작
    doc = Document()
    doc.add_heading("자동 생성 보고서", level=0)

    # PDF 열기
    pdf_fitz = fitz.open(pdf_path)
    pdf_plumber = pdfplumber.open(pdf_path)

    for page_num in range(len(pdf_fitz)):
        doc.add_heading(f'페이지 {page_num + 1}', level=1)

        # 1. 텍스트 추출
        text = pdf_plumber.pages[page_num].extract_text()
        if text:
            doc.add_paragraph(text.strip())

        # 2. 표 추출
        tables = pdf_plumber.pages[page_num].extract_tables()
        for t_idx, table in enumerate(tables):
            doc.add_paragraph(f"[표 {t_idx + 1}]", style='Intense Quote')
            if table:
                table_doc = doc.add_table(rows=1, cols=len(table[0]))
                table_doc.style = 'Table Grid'
                hdr_cells = table_doc.rows[0].cells
                for j, heading in enumerate(table[0]):
                    hdr_cells[j].text = heading if heading else ""
                for row_data in table[1:]:
                    row_cells = table_doc.add_row().cells
                    for j, cell in enumerate(row_data):
                        row_cells[j].text = cell if cell else ""

        # 3. 이미지 추출
        page = pdf_fitz[page_num]
        images = page.get_images(full=True)
        for img_index, img in enumerate(images):
            xref = img[0]
            base_image = pdf_fitz.extract_image(xref)
            image_bytes = base_image["image"]
            ext = base_image["ext"]
            image_filename = f"page_{page_num+1}_img_{img_index+1}.{ext}"
            with open(image_filename, "wb") as f:
                f.write(image_bytes)
            doc.add_paragraph(f"[이미지 {img_index+1}]")
            doc.add_picture(image_filename, width=Inches(4.5))
            os.remove(image_filename)  # 임시 이미지 파일 삭제

    pdf_plumber.close()
    pdf_fitz.close()

    # 저장
    doc.save(output_docx_path)
    print(f"[페이지별 정렬 보고서 생성 완료] {output_docx_path}")


In [6]:
extract_pdf_all_in_order("test.pdf", "output_report.docx")

Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats
Could get FontBBox from font descriptor because None cannot be parsed as 4 floats


[페이지별 정렬 보고서 생성 완료] output_report.docx


In [14]:
extract_pdf_all_in_order("testpdf.pdf", "output_report.docx")

[페이지별 정렬 보고서 생성 완료] output_report.docx


# pdf -> str 반환

In [15]:
import os
import fitz  # PyMuPDF
import pdfplumber
from io import StringIO

def extract_pdf_all_in_order_as_string(pdf_path: str) -> str:
    output = StringIO()

    output.write("# 자동 생성\n\n")

    pdf_fitz = fitz.open(pdf_path)
    pdf_plumber = pdfplumber.open(pdf_path)

    for page_num in range(len(pdf_fitz)):
        output.write(f"## 페이지 {page_num + 1}\n\n")

        # 텍스트 추출
        text = pdf_plumber.pages[page_num].extract_text()
        if text:
            output.write("**본문 텍스트:**\n")
            output.write(text.strip() + "\n\n")

        # 표 추출
        tables = pdf_plumber.pages[page_num].extract_tables()
        for t_idx, table in enumerate(tables):
            output.write(f"**[표 {t_idx + 1}]**\n")
            if table:
                for row in table:
                    row_text = " | ".join(cell if cell else "" for cell in row)
                    output.write(row_text + "\n")
                output.write("\n")

        # 이미지 추출 (실제 이미지는 안 넣고 설명만)
        page = pdf_fitz[page_num]
        images = page.get_images(full=True)
        for img_index, img in enumerate(images):
            output.write(f"[이미지 {img_index + 1}] 페이지 내 이미지 포함됨\n")

        output.write("\n---\n\n")

    pdf_plumber.close()
    pdf_fitz.close()

    return output.getvalue()
    
    # 마크다운 저장
#     markdown_text = extract_pdf_all_in_order_as_string("test.pdf")  # PDF → Markdown 문자열 생성

#     with open("output_report.md", "w", encoding="utf-8") as f:
#         f.write(markdown_text)

#     print("[저장 완료] output_report.md")

# 결과 확인
# result = extract_pdf_all_in_order_as_string("test4.pdf")
# print(result)

In [16]:
result = extract_pdf_all_in_order_as_string("testpdf.pdf")
print(result)

# 자동 생성

## 페이지 1

**본문 텍스트:**
기업맞춤형 AI-X 융복합 인재 양성 교육
1

[이미지 1] 페이지 내 이미지 포함됨
[이미지 2] 페이지 내 이미지 포함됨
[이미지 3] 페이지 내 이미지 포함됨
[이미지 4] 페이지 내 이미지 포함됨
[이미지 5] 페이지 내 이미지 포함됨
[이미지 6] 페이지 내 이미지 포함됨
[이미지 7] 페이지 내 이미지 포함됨
[이미지 8] 페이지 내 이미지 포함됨
[이미지 9] 페이지 내 이미지 포함됨
[이미지 10] 페이지 내 이미지 포함됨
[이미지 11] 페이지 내 이미지 포함됨
[이미지 12] 페이지 내 이미지 포함됨
[이미지 13] 페이지 내 이미지 포함됨
[이미지 14] 페이지 내 이미지 포함됨
[이미지 15] 페이지 내 이미지 포함됨
[이미지 16] 페이지 내 이미지 포함됨
[이미지 17] 페이지 내 이미지 포함됨
[이미지 18] 페이지 내 이미지 포함됨
[이미지 19] 페이지 내 이미지 포함됨
[이미지 20] 페이지 내 이미지 포함됨

---

## 페이지 2

**본문 텍스트:**
- -
- -
- - Q&A

[이미지 1] 페이지 내 이미지 포함됨
[이미지 2] 페이지 내 이미지 포함됨
[이미지 3] 페이지 내 이미지 포함됨
[이미지 4] 페이지 내 이미지 포함됨
[이미지 5] 페이지 내 이미지 포함됨
[이미지 6] 페이지 내 이미지 포함됨
[이미지 7] 페이지 내 이미지 포함됨
[이미지 8] 페이지 내 이미지 포함됨
[이미지 9] 페이지 내 이미지 포함됨
[이미지 10] 페이지 내 이미지 포함됨
[이미지 11] 페이지 내 이미지 포함됨
[이미지 12] 페이지 내 이미지 포함됨
[이미지 13] 페이지 내 이미지 포함됨
[이미지 14] 페이지 내 이미지 포함됨
[이미지 15] 페이지 내 이미지 포함됨
[이미지 16] 페이지 내 이미지 포함됨
[이미지 17] 페이지 내 이미지 포함됨
[이미지 18] 페이지 내 이미지 포함됨
[이미지 19] 페이지 내 이미지 포함됨
[이미지 20] 페이지 내 이미지

In [ ]:
import os
import fitz  # PyMuPDF
import pdfplumber
from io import StringIO

def extract_pdf_all_in_order_as_string(pdf_path: str) -> str:
    output = StringIO()

    output.write("# 자동 생성\n\n")

    pdf_fitz = fitz.open(pdf_path)
    pdf_plumber = pdfplumber.open(pdf_path)

    for page_num in range(len(pdf_fitz)):
        output.write(f"## 페이지 {page_num + 1}\n\n")

        # 텍스트 추출
        text = pdf_plumber.pages[page_num].extract_text()
        if text:
            output.write("**본문 텍스트:**\n")
            output.write(text.strip() + "\n\n")

        # 표 추출
        tables = pdf_plumber.pages[page_num].extract_tables()
        for t_idx, table in enumerate(tables):
            output.write(f"**[표 {t_idx + 1}]**\n")
            if table:
                for row in table:
                    row_text = " | ".join(cell if cell else "" for cell in row)
                    output.write(row_text + "\n")
                output.write("\n")

        # 이미지 추출 (실제 이미지는 안 넣고 설명만)
        page = pdf_fitz[page_num]
        images = page.get_images(full=True)
        for img_index, img in enumerate(images):
            output.write(f"[이미지 {img_index + 1}] 페이지 내 이미지 포함됨\n")

        output.write("\n---\n\n")

    pdf_plumber.close()
    pdf_fitz.close()

    return output.getvalue()
    
    # 마크다운 저장
#     markdown_text = extract_pdf_all_in_order_as_string(FILE_PATH)  # PDF → Markdown 문자열 생성

#     with open("output_report.md", "w", encoding="utf-8") as f:
#         f.write(markdown_text)

#     print("[저장 완료] output_report.md")

# 결과 확인
result = extract_pdf_all_in_order_as_string(FILE_PATH)
print(result)

# ppt

In [11]:
!pip install python-pptx


   ---------------------------------------- 0/2 [XlsxWriter]
   ---------------------------------------- 0/2 [XlsxWriter]
   -------------------- ------------------- 1/2 [python-pptx]
   -------------------- ------------------- 1/2 [python-pptx]
   -------------------- ------------------- 1/2 [python-pptx]
   ---------------------------------------- 2/2 [python-pptx]



In [2]:
%pip install python-pptx

  Using cached python_pptx-1.0.2-py3-none-any.whl.metadata (2.5 kB)
  Using cached xlsxwriter-3.2.5-py3-none-any.whl.metadata (2.7 kB)
Using cached python_pptx-1.0.2-py3-none-any.whl (472 kB)
Using cached xlsxwriter-3.2.5-py3-none-any.whl (172 kB)

   ---------------------------------------- 0/2 [XlsxWriter]
   ---------------------------------------- 0/2 [XlsxWriter]
   -------------------- ------------------- 1/2 [python-pptx]
   -------------------- ------------------- 1/2 [python-pptx]
   -------------------- ------------------- 1/2 [python-pptx]
   ---------------------------------------- 2/2 [python-pptx]

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import fitz
import pdfplumber
from pptx import Presentation
from pptx.util import Inches, Pt
from pptx.enum.shapes import MSO_SHAPE
from pptx.dml.color import RGBColor


def extract_pdf_to_pptx(pdf_path: str, pptx_output_path: str):
    prs = Presentation()
    blank_slide_layout = prs.slide_layouts[6]  # 빈 슬라이드

    pdf_fitz = fitz.open(pdf_path)
    pdf_plumber = pdfplumber.open(pdf_path)

    for page_num in range(len(pdf_fitz)):
        slide = prs.slides.add_slide(blank_slide_layout)

        # 제목 추가
        title_shape = slide.shapes.add_textbox(Inches(0.5), Inches(0.2), Inches(9), Inches(0.5))
        title_frame = title_shape.text_frame
        title_frame.text = f"페이지 {page_num + 1}"
        title_frame.paragraphs[0].font.size = Pt(24)
        title_frame.paragraphs[0].font.bold = True

        # 텍스트 추출
        text = pdf_plumber.pages[page_num].extract_text()
        if text:
            textbox = slide.shapes.add_textbox(Inches(0.5), Inches(1), Inches(8.5), Inches(3))
            tf = textbox.text_frame
            tf.text = text.strip()[:1500]  # 너무 길면 잘라냄
            tf.paragraphs[0].font.size = Pt(14)

        # 이미지 추출
        page = pdf_fitz[page_num]
        images = page.get_images(full=True)
        img_top = 4.5
        for img_index, img in enumerate(images):
            xref = img[0]
            base_image = pdf_fitz.extract_image(xref)
            image_bytes = base_image["image"]
            ext = base_image["ext"]
            image_filename = f"page_{page_num+1}_img_{img_index+1}.{ext}"
            with open(image_filename, "wb") as f:
                f.write(image_bytes)

            slide.shapes.add_picture(image_filename, Inches(1), Inches(img_top), width=Inches(6))
            img_top += 3  # 다음 이미지 위치
            os.remove(image_filename)

    pdf_fitz.close()
    pdf_plumber.close()

    prs.save(pptx_output_path)
    print(f"[PPT 생성 완료] {pptx_output_path}")


In [4]:
extract_pdf_to_pptx("testppt.pptx", "output_slides.pptx")

PdfminerException: No /Root object! - Is this really a PDF?

In [5]:
import os
from pptx import Presentation
from docx import Document
from docx.shared import Inches
from PIL import Image
from io import BytesIO

def pptx_to_docx(pptx_path: str, docx_path: str):
    prs = Presentation(pptx_path)
    doc = Document()
    doc.add_heading("PPT 보고서 자동 변환", 0)

    for i, slide in enumerate(prs.slides):
        doc.add_heading(f"슬라이드 {i + 1}", level=1)

        # 텍스트 추출
        for shape in slide.shapes:
            if shape.has_text_frame:
                text = shape.text.strip()
                if text:
                    doc.add_paragraph(text)

            # 이미지 추출
            if shape.shape_type == 13:  # picture
                image = shape.image
                image_bytes = image.blob
                image_ext = image.ext
                image_stream = BytesIO(image_bytes)
                image_path = f"slide_{i+1}_image.{image_ext}"
                
                # 저장 및 삽입
                with open(image_path, "wb") as f:
                    f.write(image_stream.getbuffer())
                doc.add_picture(image_path, width=Inches(4.5))
                os.remove(image_path)

    doc.save(docx_path)
    print(f"[변환 완료] {docx_path}")


In [6]:
pptx_to_docx("ppt1.pptx", "converted_report.docx")

[변환 완료] converted_report.docx


# ppt -> str

In [17]:
import os
from pptx import Presentation
from io import BytesIO, StringIO

def pptx_to_markdown_string(pptx_path: str) -> str:
    prs = Presentation(pptx_path)
    output = StringIO()

    output.write("# PPT 자동 변환\n\n")

    for i, slide in enumerate(prs.slides):
        output.write(f"## 슬라이드 {i + 1}\n\n")

        for shape in slide.shapes:
            # 텍스트 추출
            if shape.has_text_frame:
                text = shape.text.strip()
                if text:
                    output.write(f"{text}\n\n")

            # 이미지 추출 설명만
            if shape.shape_type == 13:  # picture
                output.write(f"[이미지 포함됨]\n\n")

        output.write("---\n\n")

    return output.getvalue()


In [18]:
result = pptx_to_markdown_string("testppt.pptx")
print(result)

# PPT 자동 변환

## 슬라이드 1

서울 PM 수요 예측 & 재배치 모델

1 조 | 조명환, 박선우, 정종혁, 김도현 | 2025.07.28

[이미지 포함됨]

[이미지 포함됨]

[이미지 포함됨]

기업맞춤형 AI-X 융복합 인재 양성 교육

[이미지 포함됨]

---

## 슬라이드 2

[이미지 포함됨]

[이미지 포함됨]

목차

---

## 슬라이드 3

[이미지 포함됨]

[이미지 포함됨]

목차

---

## 슬라이드 4

- 자전거, 전동킥보드, 전기자전거, 전동휠 등 1인 이동수단을 지칭
- 교통혼잡해결, 친환경성, 라스트마일(Last-Mile)해결 등으로 세계적으로 곽광받는 추세

퍼스널 모빌리티(Personal Mobility) 란?

[이미지 포함됨]

[이미지 포함됨]

[이미지 포함됨]

[이미지 포함됨]

퍼스널 
모빌리티

[이미지 포함됨]

[이미지 포함됨]

[이미지 포함됨]

---

## 슬라이드 5

[이미지 포함됨]

[이미지 포함됨]

[이미지 포함됨]

[이미지 포함됨]

문제 배경

[이미지 포함됨]

[이미지 포함됨]

PM 수요 불균형 현상

- 서울시 공공자전거(따릉이) 대여소별 수요 편차 심화
- 출퇴근 시간대 특정 지역 PM 부족 현상 발생
- 민간 PM 서비스(킥보드 등)도 유사한 불균형 패턴 보임
- 비효율적 재배치로 인한 운영 비용 증가

실태 분석

- 뉴스 및 언론 보도를 통한 문제 인식
- 공공데이터 분석 결과 지역별, 시간대별 수요 편차 확인
- 유동인구 데이터와 PM 대여량 간 상관관계 발견
- 기상 조건에 따른 수요 변동 가능성 확인

[이미지 포함됨]

---

## 슬라이드 6

[이미지 포함됨]

[이미지 포함됨]

[이미지 포함됨]

[이미지 포함됨]

[이미지 포함됨]

[이미지 포함됨]

[이미지 포함됨]

프로젝트 개요

수요 예측 모델 개발

[이미지 포함됨]

AI 기술을 활용한 효율적 재배치 전략 수립

- 운영 비용 최소화와 사용자 편의성 극대화
- 공공자전

In [ ]:
with open("output_from_ppt.md", "w", encoding="utf-8") as f:
    f.write(result)

In [ ]:
import os
from pptx import Presentation
from io import BytesIO, StringIO

def pptx_to_markdown_string(pptx_path: str) -> str:
    prs = Presentation(pptx_path)
    output = StringIO()

    output.write("# PPT 자동 변환\n\n")

    for i, slide in enumerate(prs.slides):
        output.write(f"## 슬라이드 {i + 1}\n\n")

        for shape in slide.shapes:
            # 텍스트 추출
            if shape.has_text_frame:
                text = shape.text.strip()
                if text:
                    output.write(f"{text}\n\n")

            # 이미지 추출 설명만
            if shape.shape_type == 13:  # picture
                output.write(f"[이미지 포함됨]\n\n")

        output.write("---\n\n")

    return output.getvalue()
    
    # 마크다운 저장
#     with open("output_from_ppt.md", "w", encoding="utf-8") as f:
#     f.write(result)

result = pptx_to_markdown_string(FILE_PATH) # ppt
print(result)

# txt

In [ ]:
def extract_txt_file_as_string(txt_path: str) -> str:
    """TXT 파일에서 전체 텍스트를 읽어 문자열로 반환"""
    with open(txt_path, "r", encoding="utf-8") as f:
        content = f.read()
    return content


if __name__ == "__main__":
    FILE_PATH = "sample.txt"  # 사용자가 올린 TXT 파일
#     OUTPUT_MD = "output_from_txt.md"
#     OUTPUT_TXT = "output_from_txt.txt"

    # 텍스트 추출
    result = extract_txt_file_as_string(FILE_PATH)

    # 출력 확인
    print(result)

    # 저장 (markdown 형식으로도 저장 가능)
#     with open(OUTPUT_MD, "w", encoding="utf-8") as f:
#         f.write("# 텍스트 파일 추출 결과\n\n")
#         f.write(result)

#     with open(OUTPUT_TXT, "w", encoding="utf-8") as f:
#         f.write(result)

#     print(f"[저장 완료] {OUTPUT_MD}, {OUTPUT_TXT}")


# 개선된 ppt

In [19]:
import os
from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE_TYPE
from io import StringIO


def pptx_to_markdown_string(pptx_path: str) -> str:
    prs = Presentation(pptx_path)
    output = StringIO()
    output.write("# PPT 자동 변환\n\n")

    def recurse_shapes(shapes, output_lines):
        for shape in shapes:
            # 텍스트 추출
            if shape.has_text_frame:
                text = shape.text.strip()
                if text:
                    output_lines.append(f"{text}\n")

            # 이미지 추출 설명
            if shape.shape_type == MSO_SHAPE_TYPE.PICTURE:
                output_lines.append("[이미지 포함됨]\n")

            # 그룹 도형 내부 재귀
            if shape.shape_type == MSO_SHAPE_TYPE.GROUP:
                recurse_shapes(shape.shapes, output_lines)

    for i, slide in enumerate(prs.slides):
        output.write(f"## 슬라이드 {i + 1}\n\n")
        slide_lines = []
        recurse_shapes(slide.shapes, slide_lines)
        output.write("".join(slide_lines))
        output.write("\n---\n\n")

    return output.getvalue()


if __name__ == "__main__":
    FILE_PATH = "testppt.pptx"
    # OUTPUT_PATH = "output_from_ppt.md"

    result = pptx_to_markdown_string(FILE_PATH)
    print(result)

    # with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    #     f.write(result)

    # print(f"[저장 완료] {OUTPUT_PATH}")


# PPT 자동 변환

## 슬라이드 1

서울 PM 수요 예측 & 재배치 모델
1 조 | 조명환, 박선우, 정종혁, 김도현 | 2025.07.28
[이미지 포함됨]
[이미지 포함됨]
[이미지 포함됨]
기업맞춤형 AI-X 융복합 인재 양성 교육
[이미지 포함됨]

---

## 슬라이드 2

[이미지 포함됨]
[이미지 포함됨]
목차
- 문제 배경
- 개발과정 개요
프로젝트 개요
01
M1 모델
M2 모델
모델개발과정
03
-   데이터 수집 개요
-   데이터 분석 인사이트
시계열 특성
데이터 수집 및 분석
02
향후추진계획
참고문헌
Q&A
발전계획
05
웹서비스 소개
웹서비스 시연
웹서비스
04

---

## 슬라이드 3

[이미지 포함됨]
[이미지 포함됨]
목차
- 문제 배경
- 개발과정 개요
프로젝트 개요
01
M1 모델
M2 모델
모델개발과정
03
-   데이터 수집 개요
-   데이터 분석 인사이트
시계열 특성
데이터 수집 및 분석
02
향후추진계획
참고문헌
Q&A
발전계획
05
웹서비스 소개
웹서비스 시연
웹서비스
04

---

## 슬라이드 4

- 자전거, 전동킥보드, 전기자전거, 전동휠 등 1인 이동수단을 지칭
- 교통혼잡해결, 친환경성, 라스트마일(Last-Mile)해결 등으로 세계적으로 곽광받는 추세
퍼스널 모빌리티(Personal Mobility) 란?
[이미지 포함됨]
[이미지 포함됨]
[이미지 포함됨]
[이미지 포함됨]
퍼스널 
모빌리티
[이미지 포함됨]
[이미지 포함됨]
[이미지 포함됨]

---

## 슬라이드 5

[이미지 포함됨]
[이미지 포함됨]
[이미지 포함됨]
[이미지 포함됨]
문제 배경
[이미지 포함됨]
[이미지 포함됨]
PM 수요 불균형 현상
- 서울시 공공자전거(따릉이) 대여소별 수요 편차 심화
- 출퇴근 시간대 특정 지역 PM 부족 현상 발생
- 민간 PM 서비스(킥보드 등)도 유사한 불균형 패턴 보임
- 비효율적 재배치로 인한 운영 비용 증가
실태 분석
- 뉴스 및 언론 보도를 통한 문제 인식
- 공공데이터 분

# 개선된 pdf

In [29]:
# pip install langchain-community pdfplumber pymupdf python-docx

import os
import fitz  # PyMuPDF
import pdfplumber
from io import StringIO

def extract_pdf_all_in_order_as_string(pdf_path: str) -> str:
    output = StringIO()

    output.write("# 자동 생성\n\n")

    pdf_fitz = fitz.open(pdf_path)
    pdf_plumber = pdfplumber.open(pdf_path)

    for page_num in range(len(pdf_fitz)):
        output.write(f"## 페이지 {page_num + 1}\n\n")

        # 텍스트 추출
        text = pdf_plumber.pages[page_num].extract_text()
        if text:
            output.write("**본문 텍스트:**\n")
            output.write(text.strip() + "\n\n")

        # 표 추출
        tables = pdf_plumber.pages[page_num].extract_tables()
        for t_idx, table in enumerate(tables):
            output.write(f"**[표 {t_idx + 1}]**\n")
            if table:
                for row in table:
                    row_text = " | ".join(cell if cell else "" for cell in row)
                    output.write(row_text + "\n")
                output.write("\n")

        # 이미지 설명만 출력
        page = pdf_fitz[page_num]
        images = page.get_images(full=True)
        for img_index, img in enumerate(images):
            output.write(f"[이미지 {img_index + 1}] 페이지 내 포함된 이미지] 실제사진이미지(그래프,텍스트아님)\n")

        output.write("\n---\n\n")

    pdf_plumber.close()
    pdf_fitz.close()

    return output.getvalue()


if __name__ == "__main__":
    FILE_PATH = "tst.pdf"  # 분석할 PDF 파일 경로
    # OUTPUT_PATH = "output_report.md"  # 저장할 마크다운 경로

    result = extract_pdf_all_in_order_as_string(FILE_PATH)

    # 콘솔에 출력
    print(result)

    # # 파일 저장을 원할 경우 주석 해제
    # with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    #     f.write(result)

    # print(f"[저장 완료] {OUTPUT_PATH}")


Cannot set gray non-stroke color because /'P21' is an invalid float value
Cannot set gray non-stroke color because /'P52' is an invalid float value
Cannot set gray non-stroke color because /'P60' is an invalid float value
Cannot set gray non-stroke color because /'P115' is an invalid float value
Cannot set gray non-stroke color because /'P144' is an invalid float value
Cannot set gray non-stroke color because /'P173' is an invalid float value
Cannot set gray non-stroke color because /'P200' is an invalid float value
Cannot set gray non-stroke color because /'P229' is an invalid float value
Cannot set gray non-stroke color because /'P256' is an invalid float value
Cannot set gray non-stroke color because /'P283' is an invalid float value
Cannot set gray non-stroke color because /'P314' is an invalid float value
Cannot set gray non-stroke color because /'P341' is an invalid float value
Cannot set gray non-stroke color because /'P372' is an invalid float value
Cannot set gray non-stroke c

# 자동 생성

## 페이지 1

**본문 텍스트:**
RNN, LSTM, Seq2Seq, Transformer
이 슬라이드에서 사용한 서체 :
-Open Sans(https://ko.cooltext.com/Download-Font-Open+Sans)
-KoPubWorld돋움체(http://www.kopus.org/biz/electronic/font.aspx)
1

**[표 1]**
 | RNN, LSTM, Seq2Seq, Transformer
이 슬라이드에서 사용한 서체 :
-Open Sans(https://ko.cooltext.com/Download-Font-Open+Sans)
-KoPubWorld돋움체(http://www.kopus.org/biz/electronic/font.aspx) |  |  |  | 
 |  |  |  |  | 
 |  |  |  |  | 
 | 1 |  |  |  | 

[이미지 1] 페이지 내 포함된 이미지] 실제사진이미지(그래프,텍스트아님)
[이미지 2] 페이지 내 포함된 이미지] 실제사진이미지(그래프,텍스트아님)
[이미지 3] 페이지 내 포함된 이미지] 실제사진이미지(그래프,텍스트아님)
[이미지 4] 페이지 내 포함된 이미지] 실제사진이미지(그래프,텍스트아님)
[이미지 5] 페이지 내 포함된 이미지] 실제사진이미지(그래프,텍스트아님)
[이미지 6] 페이지 내 포함된 이미지] 실제사진이미지(그래프,텍스트아님)
[이미지 7] 페이지 내 포함된 이미지] 실제사진이미지(그래프,텍스트아님)

---

## 페이지 2

**본문 텍스트:**
1장. 자연어처리 인공신경망
2

**[표 1]**
1장. 자연어처리 인공신경망
2

[이미지 1] 페이지 내 포함된 이미지] 실제사진이미지(그래프,텍스트아님)
[이미지 2] 페이지 내 포함된 이미지] 실제사진이미지(그래프,텍스트아님)

---

## 페이지 3

**본문 텍스트:**
1절. 순환신경망
1장. 자연어처리 인공신경망
3

**[표 1]**
 | 1절. 순환신경망
1장. 자연어처리 인공신

In [2]:
!pip install easyocr

  Using cached easyocr-1.7.2-py3-none-any.whl.metadata (10 kB)
  Using cached opencv_python_headless-4.12.0.88-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached python_bidi-0.6.6-cp310-cp310-win_amd64.whl.metadata (5.0 kB)
  Using cached shapely-2.1.1-cp310-cp310-win_amd64.whl.metadata (7.0 kB)
  Using cached pyclipper-1.3.0.post6-cp310-cp310-win_amd64.whl.metadata (9.2 kB)
  Using cached ninja-1.11.1.4-py3-none-win_amd64.whl.metadata (5.0 kB)
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   -------------------------------- ------- 2.4/2.9 MB 12.2 MB/s eta 0:00:01
   ---------------------------------------- 2.9/2.9 MB 11.1 MB/s  0:00:00
   ---------------------------------------- 0.0/38.9 MB ? eta -:--:--
   - -------------------------------------- 1.8/38.9 MB 9.1 MB/s eta 0:00:05
   ---- ----------------------------------- 4.5/38.9 MB 10.7 MB/s eta 0:00:04
   ------- -------------------------------- 7.1/38.9 MB 11.2 MB/s eta 0:00:03
   --------- ----------

In [2]:
%pip install --force-reinstall easyocr

  Using cached easyocr-1.7.2-py3-none-any.whl.metadata (10 kB)
  Using cached torch-2.7.1-cp310-cp310-win_amd64.whl.metadata (28 kB)
  Using cached torchvision-0.22.1-cp310-cp310-win_amd64.whl.metadata (6.1 kB)
  Using cached opencv_python_headless-4.12.0.88-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached scipy-1.15.3-cp310-cp310-win_amd64.whl.metadata (60 kB)
  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl.metadata (60 kB)
  Using cached pillow-11.3.0-cp310-cp310-win_amd64.whl.metadata (9.2 kB)
  Using cached scikit_image-0.25.2-cp310-cp310-win_amd64.whl.metadata (14 kB)
  Using cached python_bidi-0.6.6-cp310-cp310-win_amd64.whl.metadata (5.0 kB)
  Using cached PyYAML-6.0.2-cp310-cp310-win_amd64.whl.metadata (2.1 kB)
  Using cached shapely-2.1.1-cp310-cp310-win_amd64.whl.metadata (7.0 kB)
  Using cached pyclipper-1.3.0.post6-cp310-cp310-win_amd64.whl.metadata (9.2 kB)
  Using cached ninja-1.11.1.4-py3-none-win_amd64.whl.metadata (5.0 kB)
  Using cached filelock-3.18.0-py3

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
error: uninstall-no-record-file

Cannot uninstall numpy 1.26.4

The package's contents are unknown: no RECORD file was found for numpy.

hint: You might be able to recover from this via: pip install --force-reinstall --no-deps numpy==1.26.4


In [1]:
# pip install easyocr pdfplumber pymupdf

import os
import fitz  # PyMuPDF
import pdfplumber
from io import StringIO
import easyocr
from PIL import Image
import io

def extract_pdf_with_korean_ocr(pdf_path: str) -> str:
    output = StringIO()
    output.write("# 자동 생성 (한글 OCR 포함)\n\n")

    pdf_fitz = fitz.open(pdf_path)
    pdf_plumber = pdfplumber.open(pdf_path)
    reader = easyocr.Reader(['ko', 'en'], gpu=False)

    for page_num in range(len(pdf_fitz)):
        output.write(f"## 페이지 {page_num + 1}\n\n")

        # 일반 텍스트 추출
        text = pdf_plumber.pages[page_num].extract_text()
        if text:
            output.write("**본문 텍스트:**\n")
            output.write(text.strip() + "\n\n")

        # 표 추출
        tables = pdf_plumber.pages[page_num].extract_tables()
        for t_idx, table in enumerate(tables):
            output.write(f"**[표 {t_idx + 1}]**\n")
            for row in table:
                row_text = " | ".join(cell if cell else "" for cell in row)
                output.write(row_text + "\n")
            output.write("\n")

        # 이미지 추출 + 한글 OCR
        page = pdf_fitz[page_num]
        image_list = page.get_images(full=True)
        seen_xrefs = set()
        img_count = 0

        for img in image_list:
            xref = img[0]
            if xref in seen_xrefs:
                continue
            seen_xrefs.add(xref)

            base_image = pdf_fitz.extract_image(xref)
            image_bytes = base_image["image"]
            img_pil = Image.open(io.BytesIO(image_bytes)).convert("RGB")

            # easyocr 처리
            result = reader.readtext(np.array(img_pil), detail=0)
            ocr_text = " ".join(result)

            img_count += 1
            output.write(f"[이미지 {img_count}] OCR 결과:\n> {ocr_text.strip()}\n\n")

        output.write("---\n\n")

    pdf_fitz.close()
    pdf_plumber.close()

    return output.getvalue()


if __name__ == "__main__":
    FILE_PATH = "tst.pdf"  # 분석할 PDF 파일 경로
    OUTPUT_PATH = "output_kor_ocr.md"  # 저장할 마크다운 경로

    result = extract_pdf_with_korean_ocr(FILE_PATH)

    # 콘솔 출력
    print(result)

    # 마크다운 저장
    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        f.write(result)

    print(f"[저장 완료] {OUTPUT_PATH}")


ModuleNotFoundError: No module named 'easyocr'

# VLM PDF -> str

In [22]:
%pip install pytesseract pillow

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [ ]:
# https://github.com/UB-Mannheim/tesseract/wiki 설치 

In [26]:
import pytesseract
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

In [27]:
import fitz  # PyMuPDF
import pdfplumber
from io import StringIO
import pytesseract
from PIL import Image
import io

def extract_pdf_with_ocr(pdf_path: str) -> str:
    output = StringIO()
    output.write("# 자동 생성 (OCR 포함)\n\n")

    pdf_fitz = fitz.open(pdf_path)
    pdf_plumber = pdfplumber.open(pdf_path)

    for page_num in range(len(pdf_fitz)):
        output.write(f"## 페이지 {page_num + 1}\n\n")

        # 텍스트 추출
        text = pdf_plumber.pages[page_num].extract_text()
        if text:
            output.write("**본문 텍스트:**\n")
            output.write(text.strip() + "\n\n")

        # 표 추출
        tables = pdf_plumber.pages[page_num].extract_tables()
        for t_idx, table in enumerate(tables):
            output.write(f"**[표 {t_idx + 1}]**\n")
            for row in table:
                row_text = " | ".join(cell if cell else "" for cell in row)
                output.write(row_text + "\n")
            output.write("\n")

        # 이미지 추출 + OCR
        page = pdf_fitz[page_num]
        image_list = page.get_images(full=True)
        seen_xrefs = set()
        img_count = 0

        for img in image_list:
            xref = img[0]
            if xref in seen_xrefs:
                continue
            seen_xrefs.add(xref)

            base_image = pdf_fitz.extract_image(xref)
            image_bytes = base_image["image"]
            img_pil = Image.open(io.BytesIO(image_bytes))

            # OCR
            ocr_text = pytesseract.image_to_string(img_pil, lang='eng+kor')  # 한글 + 영어 OCR

            img_count += 1
            output.write(f"[이미지 {img_count}] 포함된 사진 → OCR 텍스트:\n> {ocr_text.strip()}\n\n")

        output.write("---\n\n")

    pdf_fitz.close()
    pdf_plumber.close()

    return output.getvalue()


In [28]:
extract_pdf_with_ocr('testpdf.pdf')

'# 자동 생성 (OCR 포함)\n\n## 페이지 1\n\n**본문 텍스트:**\n기업맞춤형 AI-X 융복합 인재 양성 교육\n1\n\n[이미지 1] 포함된 사진 → OCR 텍스트:\n> \n\n[이미지 2] 포함된 사진 → OCR 텍스트:\n> \n\n[이미지 3] 포함된 사진 → OCR 텍스트:\n> \n\n[이미지 4] 포함된 사진 → OCR 텍스트:\n> \n\n[이미지 5] 포함된 사진 → OCR 텍스트:\n> \n\n[이미지 6] 포함된 사진 → OCR 텍스트:\n> \n\n[이미지 7] 포함된 사진 → OCR 텍스트:\n> \n\n[이미지 8] 포함된 사진 → OCR 텍스트:\n> \n\n[이미지 9] 포함된 사진 → OCR 텍스트:\n> \n\n[이미지 10] 포함된 사진 → OCR 텍스트:\n> \n\n[이미지 11] 포함된 사진 → OCR 텍스트:\n> \n\n[이미지 12] 포함된 사진 → OCR 텍스트:\n> \n\n[이미지 13] 포함된 사진 → OCR 텍스트:\n> \n\n[이미지 14] 포함된 사진 → OCR 텍스트:\n> \n\n[이미지 15] 포함된 사진 → OCR 텍스트:\n> \n\n[이미지 16] 포함된 사진 → OCR 텍스트:\n> \n\n[이미지 17] 포함된 사진 → OCR 텍스트:\n> \n\n[이미지 18] 포함된 사진 → OCR 텍스트:\n> \n\n[이미지 19] 포함된 사진 → OCR 텍스트:\n> \n\n[이미지 20] 포함된 사진 → OCR 텍스트:\n> S;\n\nFlowCast\n\nWe forecast the urban flow.\n\n---\n\n## 페이지 2\n\n**본문 텍스트:**\n- -\n- -\n- - Q&A\n\n[이미지 1] 포함된 사진 → OCR 텍스트:\n> \n\n[이미지 2] 포함된 사진 → OCR 텍스트:\n> \n\n[이미지 3] 포함된 사진 → OCR 텍스트:\n> \n\n[이미지 4] 포함된 사진 → OCR 텍스트:\n> \n\n[이미지 5] 포함된 사진 → OCR 텍스

In [ ]:
# pip install pytesseract pillow pdfplumber pymupdf

import fitz  # PyMuPDF
import pdfplumber
from io import StringIO
import pytesseract
from PIL import Image
import io

def extract_pdf_with_ocr(pdf_path: str) -> str:
    output = StringIO()
    output.write("# 자동 생성 (OCR 포함)\n\n")

    pdf_fitz = fitz.open(pdf_path)
    pdf_plumber = pdfplumber.open(pdf_path)

    for page_num in range(len(pdf_fitz)):
        output.write(f"## 페이지 {page_num + 1}\n\n")

        # 텍스트 추출
        text = pdf_plumber.pages[page_num].extract_text()
        if text:
            output.write("**본문 텍스트:**\n")
            output.write(text.strip() + "\n\n")

        # 표 추출
        tables = pdf_plumber.pages[page_num].extract_tables()
        for t_idx, table in enumerate(tables):
            output.write(f"**[표 {t_idx + 1}]**\n")
            for row in table:
                row_text = " | ".join(cell if cell else "" for cell in row)
                output.write(row_text + "\n")
            output.write("\n")

        # 이미지 + OCR
        page = pdf_fitz[page_num]
        image_list = page.get_images(full=True)
        seen_xrefs = set()
        img_count = 0

        for img in image_list:
            xref = img[0]
            if xref in seen_xrefs:
                continue
            seen_xrefs.add(xref)

            base_image = pdf_fitz.extract_image(xref)
            image_bytes = base_image["image"]
            img_pil = Image.open(io.BytesIO(image_bytes))

            try:
                ocr_text = pytesseract.image_to_string(img_pil, lang='eng+kor')  # 한글 + 영어 OCR
            except Exception as e:
                ocr_text = "[OCR 실패]"

            img_count += 1
            output.write(f"[이미지 {img_count}] 포함된 사진 → OCR 텍스트:\n> {ocr_text.strip()}\n\n")

        output.write("---\n\n")

    pdf_fitz.close()
    pdf_plumber.close()

    return output.getvalue()


if __name__ == "__main__":
    FILE_PATH = "test.pdf"
    OUTPUT_PATH = "output_from_pdf_ocr.md"

    result = extract_pdf_with_ocr(FILE_PATH)
    print(result)

    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        f.write(result)

    print(f"[저장 완료] {OUTPUT_PATH}")


# VLM PPTX -> STR

In [ ]:
import os
from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE_TYPE
from io import StringIO, BytesIO
from PIL import Image
import pytesseract

# Tesseract 실행 파일 경로 설정 (설치된 경로로 바꿔줘야 함!)
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"


def pptx_to_markdown_with_ocr(pptx_path: str) -> str:
    prs = Presentation(pptx_path)
    output = StringIO()
    output.write("# PPT 자동 변환 (OCR 포함)\n\n")

    def recurse_shapes(shapes, output_lines):
        for shape in shapes:
            # 텍스트
            if shape.has_text_frame:
                text = shape.text.strip()
                if text:
                    output_lines.append(text + "\n")

            # 이미지 + OCR
            if shape.shape_type == MSO_SHAPE_TYPE.PICTURE:
                image = shape.image
                image_bytes = image.blob
                image_stream = BytesIO(image_bytes)
                img = Image.open(image_stream)

                try:
                    ocr_text = pytesseract.image_to_string(img, lang="eng+kor").strip()
                except Exception as e:
                    ocr_text = "[OCR 실패]"

                output_lines.append("[이미지 포함됨]\n")
                output_lines.append(f"> OCR 텍스트: {ocr_text}\n\n")

            # 그룹 도형 내부 탐색
            if shape.shape_type == MSO_SHAPE_TYPE.GROUP:
                recurse_shapes(shape.shapes, output_lines)

    for i, slide in enumerate(prs.slides):
        output.write(f"## 슬라이드 {i + 1}\n\n")
        slide_lines = []
        recurse_shapes(slide.shapes, slide_lines)
        output.write("".join(slide_lines))
        output.write("\n---\n\n")

    return output.getvalue()


if __name__ == "__main__":
    FILE_PATH = "input_slides.pptx"
    OUTPUT_PATH = "output_from_ppt_ocr.md"

    result = pptx_to_markdown_with_ocr(FILE_PATH)
    print(result)

    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
        f.write(result)

    print(f"[저장 완료] {OUTPUT_PATH}")
